# N113 · 数位DP、上界与前导零

**目标：** 用位位置与已用信息统计大整数区间。

**先修：** N098, N124, N023。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** tight；started；digit mask；缓存条件；区间差分；0是否计入。

## 从问题到算法

数位DP不是枚举每个整数，而是枚举数字前缀的等价状态。tight表示是否仍受上界相同前缀限制；started区分“尚未开始”的前导零和真正数字0。互异数字用mask记录已使用数字。空构造不代表正整数，所以末态需要排除0。

## 最小实现

**本章接口：** `count_special_numbers(n)`、`count_digit_one(n)`、`count_from_digit_set(digits, n)`

In [1]:
from functools import cache

def count_special_numbers(n):
    if n<=0: return 0
    digits=list(map(int,str(n)))
    @cache
    def f(i,mask,tight,started):
        if i==len(digits): return int(started)
        limit=digits[i] if tight else 9; total=0
        for d in range(limit+1):
            new_tight=tight and d==digits[i]
            if not started and d==0: total+=f(i+1,mask,new_tight,False)
            elif not mask>>d&1: total+=f(i+1,mask|1<<d,new_tight,True)
        return total
    return f(0,0,True,False)

def count_digit_one(n):
    if n<=0: return 0
    digits=list(map(int,str(n)))
    @cache
    def f(i,tight):
        if i==len(digits): return 1,0
        count=ones=0
        for d in range((digits[i] if tight else 9)+1):
            ways,tail_ones=f(i+1,tight and d==digits[i]); count+=ways; ones+=tail_ones+(d==1)*ways
        return count,ones
    return f(0,True)[1]

def count_from_digit_set(digits,n):
    if n<=0: return 0
    allowed={int(d) for d in digits}
    if not allowed<=set(range(10)): raise ValueError('single decimal digits required')
    bound=list(map(int,str(n)))
    @cache
    def f(i,tight,started):
        if i==len(bound): return int(started)
        limit=bound[i] if tight else 9; total=0
        if not started: total+=f(i+1,tight and bound[i]==0,False)
        for d in sorted(allowed):
            if d>limit or not started and d==0: continue
            total+=f(i+1,tight and d==bound[i],True)
        return total
    return f(0,True,False)

## 正确性、前提与复杂度

相同位置、约束状态与mask具有完全相同的后续选择，因此可合并。前导零不占用mask，也不要求属于允许数字集；一旦started为真，零就是实际数字。计数1的转移将“子树已有次数”与“当前为1时每条后缀贡献一次”分开相加。

**代价：** 互异数O(D·2^10·10)算术操作；数1 O(D·10)；给定集合O(D·|集合|)，缓存空间为各自状态数。D为十进制位数；此实现假定通常题目整数位数，极长数字需改迭代表。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['上界','互异数字正整数个数','数字1出现次数','仅{0,1,3}构成'],[(n,count_special_numbers(n),count_digit_one(n),count_from_digit_set(['0','1','3'],n)) for n in [0,9,20,99,135]])
print('10以内的一层mask状态：',[(d,format(1<<d,'010b')) for d in range(1,5)])

上界,互异数字正整数个数,数字1出现次数,"仅{0,1,3}构成"
0,0,0,0
9,9,1,2
20,19,12,5
99,90,20,8
135,110,70,17


10以内的一层mask状态： [(1, '0000000010'), (2, '0000000100'), (3, '0000001000'), (4, '0000010000')]


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert count_special_numbers(20)==19 and count_special_numbers(135)==110
assert count_digit_one(13)==6
assert count_special_numbers(0)==count_digit_one(0)==count_from_digit_set(['0'],20)==0
for n in range(251):
    assert count_special_numbers(n)==sum(len(set(str(x)))==len(str(x)) for x in range(1,n+1))
    assert count_digit_one(n)==sum(str(x).count('1') for x in range(n+1))
    for digits in ([],['0','1'],['1','3','5'],['0','2','9']):
        assert count_from_digit_set(digits,n)==sum(set(str(x))<=set(digits) for x in range(1,n+1))
print('N113: 所有本章断言通过')

N113: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 比较定长带前导零和不定长整数计数。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 推导区间[L,R]差分。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 2376. Count Special Integers（canonical）：[官方入口](https://leetcode.com/problems/count-special-integers/)
- 233. Number of Digit One（canonical）
- 902. Numbers At Most N Given Digit Set（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。